In [2]:
!pip install torch torchvision segmentation-models-pytorch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcugraph-cu12 24.12.0 requires pylibraft-cu12==24.12.*, but you have pylibraft-cu12 25.2.0 which is 

In [3]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp


In [7]:


# Counting functions
def count_conv2d(m, x, y):
    x = x[0]
    cin = m.in_channels // m.groups
    cout = m.out_channels // m.groups
    kh, kw = m.kernel_size
    batch_size = x.size()[0]
    kernel_mul = kh * kw * cin
    kernel_add = kh * kw * cin - 1
    bias_ops = 1 if m.bias is not None else 0
    ops = kernel_mul + kernel_add + bias_ops
    num_out_elements = y.numel()
    m.total_ops += torch.Tensor([int(num_out_elements * ops)])

def count_bn2d(m, x, y):
    x = x[0]
    nelements = x.numel()
    m.total_ops += torch.Tensor([int(2 * nelements)])

def count_relu(m, x, y):
    x = x[0]
    m.total_ops += torch.Tensor([int(x.numel())])

def count_maxpool(m, x, y):
    kernel_ops = torch.prod(torch.Tensor(m.kernel_size)) - 1
    num_elements = y.numel()
    m.total_ops += torch.Tensor([int(kernel_ops * num_elements)])

def count_avgpool(m, x, y):
    total_add = torch.prod(torch.Tensor(m.kernel_size)) - 1
    total_div = 1
    kernel_ops = total_add + total_div
    num_elements = y.numel()
    m.total_ops += torch.Tensor([int(kernel_ops * num_elements)])

def count_linear(m, x, y):
    total_mul = m.in_features
    total_add = m.in_features - 1
    num_elements = y.numel()
    m.total_ops += torch.Tensor([int((total_mul + total_add) * num_elements)])

def profile(model, input_size):
    model.eval()

    def add_hooks(m):
        if len(list(m.children())) > 0: return
        m.register_buffer('total_ops', torch.zeros(1))
        m.register_buffer('total_params', torch.zeros(1))
        for p in m.parameters():
            m.total_params += torch.Tensor([p.numel()])
        if isinstance(m, nn.Conv2d):
            m.register_forward_hook(count_conv2d)
        elif isinstance(m, nn.BatchNorm2d):
            m.register_forward_hook(count_bn2d)
        elif isinstance(m, nn.ReLU):
            m.register_forward_hook(count_relu)
        elif isinstance(m, (nn.MaxPool2d, nn.MaxPool1d, nn.MaxPool3d)):
            m.register_forward_hook(count_maxpool)
        elif isinstance(m, (nn.AvgPool2d, nn.AvgPool1d, nn.AvgPool3d)):
            m.register_forward_hook(count_avgpool)
        elif isinstance(m, nn.Linear):
            m.register_forward_hook(count_linear)

    model.apply(add_hooks)

    x = torch.zeros(input_size)
    with torch.no_grad():
        model(x)

    total_ops = 0
    total_params = 0
    for m in model.modules():
        if len(list(m.children())) > 0: continue
        total_ops += m.total_ops
        total_params += m.total_params

    return total_ops, total_params

# --- MAIN ---
if __name__ == "__main__":
    model = smp.Unet(
    encoder_name='efficientnet-b0',     
    encoder_weights='imagenet',     # Pretrained weights
    in_channels=1,                  # Input channels (e.g., RGB)
    classes=1,                      # Output channels (binary segmentation)
    activation=None                 # No activation (logits output)
    )
    checkpoint = torch.load("/kaggle/input/eff-unet-0.86/other/default/1/eff_unet_model.pth", map_location="cpu")
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        try:
            state_dict = checkpoint.module.state_dict()
        except AttributeError:
            state_dict = checkpoint.state_dict()
    
    # Clean keys if they were saved from DataParallel (prefix "module.")
    new_state_dict = {}
    for k, v in state_dict.items():
        new_key = k.replace("module.", "") if k.startswith("module.") else k
        new_state_dict[new_key] = v

    model.load_state_dict(new_state_dict)
    total_ops, total_params = profile(model, input_size=(1, 1, 256, 256))
    print(f"Total Ops: {total_ops.item()/1e9:.2f} GOps")
    print(f"Total Params: {total_params.item()/1e6:.2f} M")


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

/tmp/ipykernel_31/1413321597.py:88: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("/kaggle/input/eff-unet-0.86/other/default/1/eff_unet_model.pth", m

Total Ops: 4.99 GOps
Total Params: 2.29 M
